In [ ]:
import nest_asyncio
nest_asyncio.apply()
# 중첩 허용: 이미 돌고 있는 루프안에서 또 루프를 돌려도 괜찮게 규칙을 완화

## 비동기(async)란???

- 동기 (sync) : 일 하나가 끝나야 다음 일 시작. ex) 순서대로 줄 서기
- 비동기 (async) : 기다리는 동안 다른 일 처리. ex) 주문하고 진동벨 받고 다른 일 하기

### asyncio

- 파이썬에서 비동기를 담당하는 엔진
- 이 엔진은 한 프로세스에 하나만 실행될 수 있다.

- 간혹 ipynb에서 juputer가 이미 자체적으로 진행하고 있는 경우가 많다.
- 리마인덱스로 코드 실행 시 또 비동기를 진행시킬 수 있으므로 충돌이 나거나 오래 걸릴 수 있다.

# 데이터 로드

- RAG에 활용할 여러가지 확장자의 문서를 불러오는 방법들을 공부할 예정이다.
- 불러운 문서를 노드로 세분화 하는 방법

## 1. Doucument 객체 직접 생성

- 입력받은 택스트, 문서 고유id, 몇가지 등등의 메타데이터(데이터에 대한 정보)를 딕셔너리 형태로 가지고 있다.

In [4]:
from llama_index.core import Document # 라마인덱스에서 가장 기본적인 데이터불러오기 방법

# 예시) 제품 설명
text = '''
AI 챗봇 사용 가이드 :
- 간단한 질문으로 시작하세요.
- 구체적인 예시를 포함하면 더 좋은 답변을 받을 수 있습니다.
- 부적절한 내용은 자동으로 필터링됩니다.
'''
document = Document(
    text=text,
    metadata={'author':'제품개발팀','category':'사용설명서', 'version':'2.0'},
    id_='manual_ai_001',
)

print(document)

Doc ID: manual_ai_001
Text: AI 챗봇 사용 가이드 : - 간단한 질문으로 시작하세요. - 구체적인 예시를 포함하면 더 좋은 답변을 받을 수
있습니다. - 부적절한 내용은 자동으로 필터링됩니다.


In [7]:
from llama_index.core import SimpleDirectoryReader

# pdf 파일을 불러오기 및 documents 변수에 저장
documents = SimpleDirectoryReader(input_files=['data/pdf_example.pdf']).load_data()


In [8]:
# documents의 타입(자료형)
print(type(documents))

<class 'list'>


In [9]:
# documents의 내용
print(documents)

[Document(id_='3868ec02-1b22-4a54-9014-2b137ffaa2a9', embedding=None, metadata={'file_path': 'data\\pdf_example.pdf', 'file_name': 'pdf_example.pdf', 'file_type': 'application/pdf', 'file_size': 68851, 'creation_date': '2026-05-28', 'last_modified_date': '2026-06-03'}, excluded_embed_metadata_keys=['file_name', 'file_type', 'file_size', 'creation_date', 'last_modified_date', 'last_accessed_date'], excluded_llm_metadata_keys=['file_name', 'file_type', 'file_size', 'creation_date', 'last_modified_date', 'last_accessed_date'], relationships={}, metadata_template='{key}: {value}', metadata_separator='\n', text_resource=MediaResource(embeddings=None, data=None, text='%PDF-1.7\r\n%\r\n1 0 obj\r\n<</Type/Catalog/Pages 2 0 R/Lang(ko) /StructTreeRoot 15 0 R/MarkInfo<</Marked true>>/Metadata 292 0 R/ViewerPreferences 293 0 R>>\r\nendobj\r\n2 0 obj\r\n<</Type/Pages/Count 1/Kids[ 3 0 R] >>\r\nendobj\r\n3 0 obj\r\n<</Type/Page/Parent 2 0 R/Resources<</Font<</F1 5 0 R/F2 12 0 R>>/ExtGState<</GS10 10

In [10]:
print(documents[0])

Doc ID: 3868ec02-1b22-4a54-9014-2b137ffaa2a9
Text: %PDF-1.7  %  1 0 obj  <</Type/Catalog/Pages 2 0 R/Lang(ko)
/StructTreeRoot 15 0 R/MarkInfo<</Marked true>>/Metadata 292 0
R/ViewerPreferences 293 0 R>>  endobj  2 0 obj  <</Type/Pages/Count
1/Kids[ 3 0 R] >>  endobj  3 0 obj  <</Type/Page/Parent 2 0
R/Resources<</Font<</F1 5 0 R/F2 12 0 R>>/ExtGState<</GS10 10 0 R/GS11
11 0 R>>/ProcSet[/PDF/Text...


In [ ]:
document = documents[0] # 실제 읽은 내용을 변수에 저장

# 문서 내용 확인
print(document.text)

In [13]:
# 파일 정보 확인
print(document.metadata['file_name'])
print(document.metadata['file_type'])

pdf_example.pdf
application/pdf


In [14]:
# 문서 id 확인
print(document.id_)

3868ec02-1b22-4a54-9014-2b137ffaa2a9


## 3. 노드

- 라마인덱스에서문서를 작은 단위로 나눈 기본 데이터 구조
- 검색와 질의응답을 효율적으로 수행할 수 있도록 도와준다.
- 특정 질문에 대해 관련성이 높은 부분만 검색하여 LLM이 보다 정확한 응답을 생성할 수 있다.

## Node 객체 직접 생성

In [17]:

from llama_index.core import Document
from llama_index.core.schema import TextNode

document = Document(
    text="""인공지능은 우리의 미래를 변화시킬 것입니다.
            이러한 변화에 우리는 준비되어 있어야 합니다."""
)

# 문서를 두 개의 노드로 분할 (각 문장을 하나의 노드로)
node1 = TextNode(text=document.text[:24], doc_id=document.id_)
node2 = TextNode(text=document.text[25:], doc_id=document.id_)

print(node1)
print(node2)

Node ID: 37f6ceeb-6a03-472b-aba6-fdbbb8917d7c
Text: 인공지능은 우리의 미래를 변화시킬 것입니다.
Node ID: d392e56b-7a11-430a-8c3d-e7010d9a37c9
Text: 이러한 변화에 우리는 준비되어 있어야 합니다.


## 5. 문서를 불러와서 Document와 Node 객체 생성

- 노드파서 클래스 (SimpleNodeParser) : Document를 노드로 나눌때 사용되는 클래스
    - 파서 (Parser) : 큰 정보를 의미 있는 조각들로 나누는 것 

In [29]:
from llama_index.core import SimpleDirectoryReader
from llama_index.core.node_parser import SimpleNodeParser

# docx 파일 불러오기 및 documents 변수에 저장 (data폴더 안 docx확장자 파일 불러오기)

documents = SimpleDirectoryReader(input_dir='data', required_exts=['.docx']).load_data()

In [23]:
documents

[Document(id_='d25072fe-b7ec-4ded-b197-03d7d34a01c1', embedding=None, metadata={'file_path': 'c:\\Users\\Administrator\\ai_class\\llm_labs\\data\\word_example.docx', 'file_name': 'word_example.docx', 'file_type': 'application/vnd.openxmlformats-officedocument.wordprocessingml.document', 'file_size': 15420, 'creation_date': '2026-05-28', 'last_modified_date': '2026-06-03'}, excluded_embed_metadata_keys=['file_name', 'file_type', 'file_size', 'creation_date', 'last_modified_date', 'last_accessed_date'], excluded_llm_metadata_keys=['file_name', 'file_type', 'file_size', 'creation_date', 'last_modified_date', 'last_accessed_date'], relationships={}, metadata_template='{key}: {value}', metadata_separator='\n', text_resource=MediaResource(embeddings=None, data=None, text='PK\x03\x04\x14\x00\x06\x00\x08\x00\x00\x00!\x00ߤlZ\x01\x00\x00 \x05\x00\x00\x13\x00\x08\x02[Content_Types].xml \x04\x02(\x00\x02\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x0

- chunk(청크) : 파서를 통해 나누어진 조각, 노드와 비슷한 의미
-

In [31]:
parser = SimpleNodeParser.from_defaults(
    chunk_size=200, # 200토큰을 기준으로 문서의 내용을 나눈다.
    chunk_overlap=20  # 20토큰씩 중복해서 겹쳐라.
)

nodes = parser.get_nodes_from_documents([documents[0]])

In [32]:
nodes

[TextNode(id_='bc124231-b4f1-4c06-8d9d-4e589fb112a7', embedding=None, metadata={'file_path': 'c:\\Users\\Administrator\\ai_class\\llm_labs\\data\\word_example.docx', 'file_name': 'word_example.docx', 'file_type': 'application/vnd.openxmlformats-officedocument.wordprocessingml.document', 'file_size': 15420, 'creation_date': '2026-05-28', 'last_modified_date': '2026-06-03'}, excluded_embed_metadata_keys=['file_name', 'file_type', 'file_size', 'creation_date', 'last_modified_date', 'last_accessed_date'], excluded_llm_metadata_keys=['file_name', 'file_type', 'file_size', 'creation_date', 'last_modified_date', 'last_accessed_date'], relationships={<NodeRelationship.SOURCE: '1'>: RelatedNodeInfo(node_id='203cde1a-601e-431a-a0ba-83076894209b', node_type=<ObjectType.DOCUMENT: '4'>, metadata={'file_path': 'c:\\Users\\Administrator\\ai_class\\llm_labs\\data\\word_example.docx', 'file_name': 'word_example.docx', 'file_type': 'application/vnd.openxmlformats-officedocument.wordprocessingml.document

In [ ]:
# 노드 확인
for i, node in enumerate(nodes):
    print(f'=== Node {i+1} ===')
    print(f'Text : {node.text}')
    print()

In [36]:
# 메타데이터 확인
print(node.metadata)

{'file_path': 'c:\\Users\\Administrator\\ai_class\\llm_labs\\data\\word_example.docx', 'file_name': 'word_example.docx', 'file_type': 'application/vnd.openxmlformats-officedocument.wordprocessingml.document', 'file_size': 15420, 'creation_date': '2026-05-28', 'last_modified_date': '2026-06-03'}


## 6. 특정 폴더(디렉토리) 안의 모든 파일 불러오기(로드)

In [37]:
from llama_index.core import SimpleDirectoryReader

documents = SimpleDirectoryReader('./data').load_data()


In [39]:
# 디랙토리 내 파일 개수
len(documents) # 폴더는 나오지 않는다

10

In [ ]:
for d in documents:
    print(d.metadata['file_name'])

In [41]:
# 하위 폴더를 포함
documents = SimpleDirectoryReader(input_dir='./data', recursive=True).load_data()

# 하위 폴더 포함 파일 개수
len(documents)

17

In [42]:
for d in documents:
    print(d.metadata['file_name'])

data_level0.bin
header.bin
length.bin
link_lists.bin
chroma.sqlite3
csv_example.csv
excel_example.xlsx
html_example.html
img_example.jpg
json_example.json
markdown_example.md
pdf_example.pdf
240828_(AI리포트)_미국의_인공지능(AI)_정책,전략.pdf
NIPS-2017-attention-is-all-you-need-Paper.pdf
txt_example.txt
word_example.docx
xml_example.xml


In [ ]:
# 특정 파일만 불러오기
documents = SimpleDirectoryReader(input_dir='./data',
                                  input_files=['./data/json_example.json',
                                               './data/txt_example.txt'])

In [44]:
documents

In [45]:
# 특정 파일 제외
documents = SimpleDirectoryReader(input_dir='./data',
                                  exclude=['./data/json_example.json',
                                               './data/txt_example.txt'])

In [46]:
# 특정 확장자만 불러오기
documents = SimpleDirectoryReader(input_dir='./data',
                                  required_exts=['.txt'])

## 7. 엑셀 파일만 불러오기

In [47]:
documents = SimpleDirectoryReader(input_dir='./data',
                                  required_exts=['.xlsx']).load_data()

len(documents)

1

In [48]:
# 엑셀 파일 안 내용
documents[0].text

'PK\x03\x04\x14\x00\x00\x00\x08\x00\x00\x00?\x00a]I:O\x01\x00\x00\x04\x00\x00\x13\x00\x00\x00[Content_Types].xmln0\x10E*1tQU\x15E\x1f\x16\x03\\{B,\x1c\x0c\x14P[QMdsǎ<\x18-\x1b- \r\x14\'2:\x18맥x<w"CR(\x17<b\x05(Fëd\x15\x013nX(KFa\x11"xT!55MeTz oz[\'S!GQ \t\x1ca-lYP1:\x15q].E7;;\r65\x0bKh+\x7f\x036}3\x1a*ыjX%M\x14"J\x17]\x0cUe5Ǽ\x02@\x06L\x1e\x12\x12Ye>!=jO$.DZ9GŘ@\x19\x01q\x08\x7f\x7f69\x02\x0ci\x11ök(Owbr?\x1dy\x127J|\r\\{os>\x19~\x01PK\x03\x04\x14\x00\x00\x00\x08\x00\x00\x00?\x00I\x00\x00\x00K\x02\x00\x00\x0b\x00\x00\x00_rels/.relsN0\x0c@|EnH\x08 &4>$n\x1bă\uf250@\x0ci\x07qg<R\x1c\x0c,\x1a\x14\x05·iq\x0f*\x0b\x06#\x072p\x0cfL#JɽY\x15H\x06zu=M+\x14OiB)vtɀ愩@ں%15ln[oa gZ(dL\x1dy\uf706W*P]V\u05fb=HСh\x11SNZu\x1c]\tόKB\x1c#wY\tc2\'7|\x00PK\x03\x04\x14\x00\x00\x00\x08\x00\x00\x00?\x00Du[\x00\x00\x00\x02\x00\x00\x1a\x00\x00\x00xl/_rels/workbook.xml.relsj0\x10D\nZv\x12J)s)\\\x03LlIhi\x11\tM\x1d\x08\x07Čؙ\x07Ћ\x03&WP\x15%\x08&η\n>wo \x0f\x1e\x15H\x1fk3H"xR㋔d\x1c\x0e\x10&AsQnQ.Ii\x06Wbk\x15@ƈ\x

## 8. 이미지 파일만 불러오기

In [49]:
documents = SimpleDirectoryReader(input_dir='./data',
                                  required_exts=['.jpg', '.jpeg', '.png']).load_data()

In [50]:
# 이미지 파일 개수
len(documents)

1

In [51]:
# 이미지 실제로 출력
from IPython.display import Image

Image(url=documents[0].metadata['file_path'])

## 돌발퀴즈!

-data폴더 안 csv파일만 불러와 보자. 변수명은 documents, 파일 개수, 파일 안 내용

-data폴더 안 xml파일만 불러와 보자. 변수명은 documents, 파일 개수, 파일 안 내용

In [58]:
documents = SimpleDirectoryReader(input_dir='./data',
                                  required_exts=['.csv']).load_data()

len(documents)

1

In [ ]:
documents = SimpleDirectoryReader(input_dir='./data',
                                  required_exts=['.xml']).load_data()

len(documents)